# F1 Race Prediction Model

End-to-end notebook: fetch data → EDA → train models → predict.

**Data source:** [f1api.dev](https://f1api.dev) (free, open source, replaces deprecated Ergast)

**Prediction targets:**
- `points_finish` — will a driver finish in the top 10?
- `position_target` — what position will a driver finish?


In [1]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from f1_data import get_f1_data
from f1_models import train_f1_models, FEATURE_COLS
from f1_predictor import F1RacePredictor

warnings.filterwarnings('ignore')
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Fetch Data

In [2]:
# Fetches ~10 years of race results from f1api.dev
# This will take a few minutes due to rate-limiting pauses
df = get_f1_data(start_year=2015, end_year=2024)
print(f'Shape: {df.shape}')
df.head()

F1 DATA PIPELINE (2015-2024)

2015 (19 rounds)
  R01... 

AttributeError: 'list' object has no attribute 'get'

## 2. Exploratory Data Analysis

In [ ]:
print('Dataset summary')
print(f"  Years:         {df['year'].min()}–{df['year'].max()}")
print(f"  Unique drivers: {df['driver_id'].nunique()}")
print(f"  Constructors:  {df['constructor_id'].nunique()}")
print(f"  Points finish rate: {df['points_finish'].mean():.1%}")

In [ ]:
# How strongly does grid position predict a points finish?
grid_pts = (
    df.groupby('grid_position')['points_finish']
    .agg(['mean', 'count'])
    .query('count > 20')
)

fig, axes = plt.subplots(1, 2)

axes[0].bar(grid_pts.index, grid_pts['mean'], color='steelblue')
axes[0].set(title='Grid Position vs Points Finish Rate',
            xlabel='Grid Position', ylabel='P(points finish)')

valid = df[df['final_position'] <= 20]
axes[1].hist(valid['final_position'], bins=20, edgecolor='black', color='coral', alpha=0.8)
axes[1].set(title='Distribution of Finishing Positions (excl. DNF)',
            xlabel='Finish Position', ylabel='Count')

plt.tight_layout()
plt.show()

In [ ]:
# Feature correlations with the binary target
corr = df[FEATURE_COLS + ['points_finish']].corr()['points_finish'].drop('points_finish').sort_values()

plt.figure(figsize=(8, 5))
corr.plot(kind='barh', color=['coral' if v < 0 else 'steelblue' for v in corr])
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Feature Correlation with Points Finish')
plt.tight_layout()
plt.show()

## 3. Train Models

In [ ]:
manager = train_f1_models(df)
manager.save_models('./models')

In [ ]:
print('Model summary:')
for target, res in manager.results.items():
    print(f"  {target:20s}  {res['model']:25s}  {res['metric']}={res['score']:.4f}")

## 4. Predictions

In [ ]:
predictor = F1RacePredictor('./models')

# Example: a front-running driver vs a midfield driver
drivers = [
    {
        'driver_name': 'Front Runner',
        'grid_position': 2,
        'driver_points_5race_avg': 20.0,
        'driver_points_10race_avg': 18.5,
        'driver_finish_rate_5race': 0.92,
        'driver_grid_avg_5race': 2.5,
        'driver_career_wins': 30,
        'driver_races_completed': 180,
        'constructor_points_5race_avg': 25.0,
        'constructor_dnf_rate_5race': 0.04,
        'driver_circuit_avg_finish': 2.0,
    },
    {
        'driver_name': 'Midfield Driver',
        'grid_position': 12,
        'driver_points_5race_avg': 4.0,
        'driver_points_10race_avg': 3.5,
        'driver_finish_rate_5race': 0.70,
        'driver_grid_avg_5race': 13.0,
        'driver_career_wins': 0,
        'driver_races_completed': 60,
        'constructor_points_5race_avg': 8.0,
        'constructor_dnf_rate_5race': 0.14,
        'driver_circuit_avg_finish': 12.0,
    },
]

result = predictor.predict_race(drivers)

print('\n=== PREDICTIONS ===')
for p in result['predictions']:
    print(
        f"Grid {p['grid_position']:2d} → Pos {p['predicted_position']:2d}  "
        f"{p['driver_name']:<20s}  "
        f"Points prob: {p['points_probability']:.1%}"
    )

In [6]:
df = get_f1_data(2015, 2025)
model_mgr = train_f1_models(df)
model_mgr.save_models('./models')

F1 DATA PIPELINE (2015-2025)

2015 (19 rounds)
  R01... 

AttributeError: 'list' object has no attribute 'get'